In [ ]:
import pandas as pd
import re
import os
import glob
import unicodedata
import nltk
from nltk.corpus import stopwords
import simplemma

nltk.download('stopwords', quiet=True)

TR_EXTRA_FUNCTIONAL = {
    'bir', 'iki', 'üç', 'tüm', 'tüm', 'olan', 'oldu', 'olduğu', 'olmuş', 'olmak',
    'yani', 'işte', 'şey', 'şu', 'bunu', 'şunu', 'onu', 'bana', 'sana', 'ona',
    'bizi', 'sizi', 'onları', 'benim', 'senin', 'onun', 'bizim', 'sizin', 'onların',
    'kendi', 'kendisi', 'kendine', 'kendini', 'kendim', 'kendin',
    'sonra', 'önce', 'şimdi', 'bugün', 'yarın', 'dün', 'hala', 'henüz', 'artık',
    'sadece', 'yalnız', 'yalnızca', 'ancak', 'fakat', 'lakin', 'oysa', 'halbuki',
    'çünkü', 'zira', 'madem', 'mademki', 'eğer', 'şayet',
    'mı', 'mi', 'mu', 'mü', 'ki', 'ya', 'yada', 'veya', 'veyahut',
    'pek', 'hiçbir', 'birçok', 'bazen', 'genellikle', 'genelde', 'özellikle',
    'tamamen', 'tam', 'tabi', 'tabii', 'evet', 'hayır', 'belki',
    'aynı', 'farklı', 'başka', 'diğer',
    'falan', 'filan', 'felan',
    'yine', 'gene', 'tekrar',
    'kez', 'defa', 'kere',
    'gerek', 'gerekir', 'gerekiyor', 'lazım',
    'var', 'yok', 'oldu', 'olur', 'olmaz', 'olabilir',
    'biraz', 'birazcık', 'azıcık',
    'olarak', 'rağmen', 'göre', 'kadar', 'doğru', 'beri',
    'üzerine', 'üzerinde', 'altına', 'altında', 'içine', 'içinde', 'dışına', 'dışında',
    'önüne', 'önünde', 'arkasına', 'arkasında', 'yanına', 'yanında',
    'şöyle', 'böyle', 'öyle', 'nasıl', 'niçin', 'neden', 'nerede', 'nereye', 'ne zaman',
    'kim', 'kime', 'kimi', 'kimin', 'hangi',
    've', 'ile', 'de', 'da', 'mi', 'mı', 'mu', 'mü',
}

TR_DOMAIN_STOP = {
    'film', 'filmi', 'filmin', 'filme', 'filmler', 'filmleri', 'filmlerin',
    'sinema', 'sinemanın', 'sinemada',
    'izle', 'izledim', 'izlemek', 'izleyen', 'izleyici', 'izleme',
    'oyuncu', 'oyuncusu', 'oyuncular', 'oyunculuk',
    'yönetmen', 'yönetmeni', 'yönetmenin',
    'sahne', 'sahnesi', 'sahneler', 'sahnede',
    'yapım', 'yapımı', 'yapımcı',
    'senaryo', 'senaryosu',
    'karakter', 'karakteri', 'karakterler',
    'rol', 'rolü', 'roller',
    'bence', 'kanımca', 'kanaatimce',
    'spoiler', 'bkz',
}

EN_DOMAIN_STOP = {
    'film', 'films', 'movie', 'movies', 'cinema', 'cinematic',
    'watch', 'watching', 'watched', 'watcher', 'viewer', 'viewing',
    'actor', 'actress', 'actors', 'actresses', 'acting', 'cast',
    'director', 'directing', 'directed',
    'scene', 'scenes', 'shot', 'shots',
    'character', 'characters',
    'role', 'roles',
    'story', 'plot', 'screenplay', 'script',
    'see', 'seen', 'seeing', 'saw',
    'really', 'pretty', 'quite', 'just', 'also', 'even', 'still', 'much', 'many',
    'one', 'two', 'first', 'last',
    'get', 'got', 'getting', 'make', 'made', 'making',
    'thing', 'things', 'something', 'anything', 'nothing', 'everything',
    'way', 'time', 'times',
}

stop_words_tr = set(stopwords.words('turkish')) | TR_EXTRA_FUNCTIONAL | TR_DOMAIN_STOP
stop_words_en = set(stopwords.words('english')) | EN_DOMAIN_STOP

print(f'TR stopword sayısı: {len(stop_words_tr)} (NLTK 53 + extra)')
print(f'EN stopword sayısı: {len(stop_words_en)} (NLTK + domain)')

_lemma_cache_tr: dict[str, str] = {}
_lemma_cache_en: dict[str, str] = {}

def lemmatize_word(word: str, lang: str) -> str:
    cache = _lemma_cache_tr if lang == 'tr' else _lemma_cache_en
    if word in cache:
        return cache[word]
    try:
        lemma = simplemma.lemmatize(word, lang=lang)
    except Exception:
        lemma = word
    cache[word] = lemma
    return lemma

raw_data_dir = 'raw/'
processed_data_dir = 'processed/'
os.makedirs(processed_data_dir, exist_ok=True)

MIN_WORD_COUNT = 5 

TR stopword sayısı: 214 (NLTK 53 + extra)
EN stopword sayısı: 262 (NLTK + domain)


In [ ]:
def clean_text(text, lang='tr'):
    """Lowercase + NFC + HTML/URL temizliği + harf-dışı silme + lemmatize + stopword.
    review_text'i bozmaz; sadece cleaned_text üretmek için kullanılır."""
    if not isinstance(text, str):
        return ''

    text = unicodedata.normalize('NFC', text)

    if lang == 'tr':
        text = text.replace('İ', 'i').replace('I', 'ı')
    text = text.lower()

    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+', ' ', text)

    text = re.sub(r'[^a-zçğıöşü\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if not text:
        return ''

    sw = stop_words_tr if lang == 'tr' else stop_words_en
    out = []
    for w in text.split():
        if len(w) < 2:
            continue
        lemma = lemmatize_word(w, lang)
        if w in sw or lemma in sw:
            continue
        out.append(lemma)

    return ' '.join(out)

In [ ]:
dfs = []
files = sorted(glob.glob(os.path.join(raw_data_dir, '*_reviews.csv')))

files = [f for f in files if 'top250' not in os.path.basename(f)]

for f in files:
    source_name = os.path.basename(f).replace('_reviews.csv', '')
    df = pd.read_csv(f)

    df = df.dropna(subset=['review_text']).copy()

    before = len(df)
    df = df.drop_duplicates(subset=['imdb_id', 'review_text']).copy()
    intra_dup = before - len(df)

    df['language'] = 'en' if source_name in ('imdb', 'letterboxd') else 'tr'

    print(f'{source_name}: {len(df):,} yorum işleniyor (kaynak içi {intra_dup:,} yineleme atıldı)')
    df['cleaned_text'] = df.apply(lambda r: clean_text(r['review_text'], r['language']), axis=1)

    df['_wc'] = df['cleaned_text'].str.split().str.len()
    before_wc = len(df)
    df = df[df['_wc'] >= MIN_WORD_COUNT].drop(columns='_wc').copy()
    print(f'  → kelime sayısı < {MIN_WORD_COUNT} olan {before_wc - len(df):,} yorum atıldı, kalan {len(df):,}')

    df = df[['imdb_id', 'source', 'language', 'review_text', 'cleaned_text']]
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

before = len(final_df)
final_df = final_df.drop_duplicates(subset=['imdb_id', 'cleaned_text']).reset_index(drop=True)
cross_dup = before - len(final_df)
print(f'\nKaynaklar arası yinelen (imdb_id + cleaned_text bazlı) atılan: {cross_dup:,}')
print(f'Toplam temiz satır: {len(final_df):,}')

beyazperde: 27,014 yorum işleniyor (kaynak içi 4 yineleme atıldı)
  → kelime sayısı < 5 olan 2,545 yorum atıldı, kalan 24,469
eksisozluk: 111,706 yorum işleniyor (kaynak içi 60 yineleme atıldı)
  → kelime sayısı < 5 olan 8,558 yorum atıldı, kalan 103,148
imdb: 154,611 yorum işleniyor (kaynak içi 2,031 yineleme atıldı)
  → kelime sayısı < 5 olan 1,747 yorum atıldı, kalan 152,864
letterboxd: 211,382 yorum işleniyor (kaynak içi 133 yineleme atıldı)
  → kelime sayısı < 5 olan 23,386 yorum atıldı, kalan 187,996
sinemalar: 132,645 yorum işleniyor (kaynak içi 907 yineleme atıldı)
  → kelime sayısı < 5 olan 20,012 yorum atıldı, kalan 112,633

Kaynaklar arası yinelen (imdb_id + cleaned_text bazlı) atılan: 2,199
Toplam temiz satır: 578,911


In [ ]:
print(f'Toplam Yorum Sayısı: {len(final_df):,}')
print(final_df['language'].value_counts())
print()
print(final_df['source'].value_counts())
print()
final_df.head(3)

Toplam Yorum Sayısı: 578,911
language
en    339859
tr    239052
Name: count, dtype: int64

source
letterboxd    187183
imdb          152676
sinemalar     111577
eksisozluk    103012
beyazperde     24463
Name: count, dtype: int64



,imdb_id,source,language,review_text,cleaned_text
0,tt0111161,beyazperde,tr,SİNEMA TARİHİNİN KİLOMETRE TAŞI BİR FİLM Dünya...,tarih kilometre taş dünya tarih gel geçmiş iyi...
1,tt0111161,beyazperde,tr,"Tartışmasız,gelmiş geçmiş en iyi film ve bunda...",tartış gel geçmiş iyi iyi yap san yeşil yol ka...
2,tt0111161,beyazperde,tr,Sinema tarihinin gelmiş geçmiş en büyük dramas...,tarih gel geçmiş büyük drama baba dan iyi imdb...


In [ ]:

import numpy as np

MIN_YORUM_DIL_BASINA = 100
MAX_ORANSAL_FARK = 0.05
RANDOM_STATE = 42

kept_indices = []
film_tam_denge = 0  
film_orneklem = 0  
film_dustu = 0    

for imdb_id, grp in final_df.groupby('imdb_id', sort=False):
    tr_rows = grp[grp['language'] == 'tr']
    en_rows = grp[grp['language'] == 'en']
    n_tr, n_en = len(tr_rows), len(en_rows)
    if n_tr < MIN_YORUM_DIL_BASINA or n_en < MIN_YORUM_DIL_BASINA:
        film_dustu += 1
        continue
    mx = max(n_tr, n_en)
    rel = abs(n_tr - n_en) / mx
    if rel <= MAX_ORANSAL_FARK:
        kept_indices.extend(grp.index.tolist())
        film_tam_denge += 1
    else:
        m = min(n_tr, n_en)
        idx_tr = tr_rows.sample(n=m, random_state=RANDOM_STATE).index
        idx_en = en_rows.sample(n=m, random_state=RANDOM_STATE).index
        kept_indices.extend(idx_tr.tolist())
        kept_indices.extend(idx_en.tolist())
        film_orneklem += 1

final_df_merged = final_df.loc[kept_indices].copy()

print('--- Film bazlı birleştirme özeti (reviews_merged) ---')
print(f'Min dil başına: {MIN_YORUM_DIL_BASINA} | %5 eşiği: zaten dengeli filmlerde tüm satırlar; değilse min sayıya kırpma')
print(f'Toplam imdb_id (ham): {final_df["imdb_id"].nunique():,}')
print(f'  Min yorum şartını geçemeyen film: {film_dustu:,}')
print(f'  Dahil edilen film: {film_tam_denge + film_orneklem:,}  (bunlardan {film_tam_denge:,} zaten ≤%5 fark, {film_orneklem:,} alt örneklemeli)')
print(f'Tam veri satırı: {len(final_df):,} → Birleştirilmiş satır: {len(final_df_merged):,}')
print(f'TR: {(final_df_merged["language"] == "tr").sum():,} | EN: {(final_df_merged["language"] == "en").sum():,}')

--- Film bazlı birleştirme özeti (reviews_merged) ---
Min dil başına: 100 | %5 eşiği: zaten dengeli filmlerde tüm satırlar; değilse min sayıya kırpma
Toplam imdb_id (ham): 250
  Min yorum şartını geçemeyen film: 21
  Dahil edilen film: 229  (bunlardan 4 zaten ≤%5 fark, 225 alt örneklemeli)
Tam veri satırı: 578,911 → Birleştirilmiş satır: 382,977
TR: 191,415 | EN: 191,562


In [6]:
# Tam birleşik veri + analiz alt kümesi
out_all = os.path.join(processed_data_dir, 'all_reviews_cleaned.csv')
final_df.to_csv(out_all, index=False, encoding='utf-8-sig')
print(f'Tam veri kaydedildi: {out_all}')

out_merged = os.path.join(processed_data_dir, 'reviews_merged.csv')
final_df_merged.to_csv(out_merged, index=False, encoding='utf-8-sig')
print(f'Film filtreli analiz verisi kaydedildi: {out_merged}')

Tam veri kaydedildi: processed/all_reviews_cleaned.csv
Film filtreli analiz verisi kaydedildi: processed/reviews_merged.csv
